<a href="https://colab.research.google.com/github/PutriBalqisAfradinata/BigData26_A_2411531009_PutriBalqisAfradinata/blob/main/Praktikum4/BD_P04_Notebook_Lengkap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRAKTIKUM 4 NoSQL DAN PEMODELAN DATA

# K-1. Menyiapkan Lingkungan

In [1]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.7 MB/s eta 0:00:00


In [6]:
from google.colab import drive
drive.mount("/content/drive")

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
TRIPS_BERSIH = f"{DIR_P3}/trips_bersih"

import os
os.makedirs(DIR_SIMPAN, exist_ok=True)
assert os.path.exists(TRIPS_BERSIH), "Folder trips_bersih/ tidak ada. Jalankan ulang P3/K-10 atau pakai K-1b."

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import zipfile

ZIP_P3 = "/content/drive/MyDrive/BigData/Praktikum3/trips_bersih.zip"
TRIPS_BERSIH = "/content/drive/MyDrive/BigData/Praktikum3/trips_bersih"

# Ekstrak jika folder trips_bersih belum ada
if not os.path.exists(TRIPS_BERSIH):
    print("Folder trips_bersih belum ada. Mengekstrak ZIP...")

    with zipfile.ZipFile(ZIP_P3, "r") as zip_ref:
        zip_ref.extractall("/content/drive/MyDrive/BigData/Praktikum3")

    print("Ekstraksi selesai.")
else:
    print("Folder trips_bersih sudah ada. Tidak perlu ekstrak.")

print("\nCek folder:")
print(os.listdir("/content/drive/MyDrive/BigData/Praktikum3"))

Folder trips_bersih belum ada. Mengekstrak ZIP...
Ekstraksi selesai.

Cek folder:
['laporan_kualitas.csv', 'lineage.csv', 'pengukuran_kinerja.csv', 'trips_bersih.zip', 'kamus_data.md', 'borough_naik=Bronx', 'borough_naik=Brooklyn', 'borough_naik=EWR', 'borough_naik=Manhattan', 'borough_naik=Queens', 'borough_naik=Staten%20Island', 'borough_naik=Unknown', 'borough_naik=__HIVE_DEFAULT_PARTITION__']


In [5]:
import os
import shutil

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
TRIPS_BERSIH = os.path.join(DIR_P3, "trips_bersih")

# Buat folder trips_bersih
os.makedirs(TRIPS_BERSIH, exist_ok=True)

# Pindahkan folder hasil partisi ke dalam trips_bersih
for nama in os.listdir(DIR_P3):
    sumber = os.path.join(DIR_P3, nama)

    if os.path.isdir(sumber) and nama.startswith("borough_naik="):
        tujuan = os.path.join(TRIPS_BERSIH, nama)

        if not os.path.exists(tujuan):
            shutil.move(sumber, tujuan)

print("Struktur data sudah dirapikan.")
print("\nIsi trips_bersih:")
print(os.listdir(TRIPS_BERSIH))

Struktur data sudah dirapikan.

Isi trips_bersih:
['borough_naik=Bronx', 'borough_naik=Brooklyn', 'borough_naik=EWR', 'borough_naik=Manhattan', 'borough_naik=Queens', 'borough_naik=Staten%20Island', 'borough_naik=Unknown', 'borough_naik=__HIVE_DEFAULT_PARTITION__']


In [8]:
import os, time, json, sqlite3, random, shutil, glob
import numpy as np, pandas as pd

# Cari semua file parquet di dalam trips_bersih
file_parquet = glob.glob(
    os.path.join(TRIPS_BERSIH, "**", "*.parquet"),
    recursive=True
)

print("Jumlah file Parquet:", len(file_parquet))

# Baca file satu per satu agar schema antar-partisi tidak bentrok
bagian_data = []

for file in file_parquet:
    try:
        bagian = pd.read_parquet(file, engine="pyarrow")

        # Ambil nilai borough dari nama folder partisi
        folder = os.path.basename(os.path.dirname(file))

        if folder.startswith("borough_naik="):
            borough = folder.replace("borough_naik=", "")

            if "borough_naik" not in bagian.columns:
                bagian["borough_naik"] = borough

        bagian_data.append(bagian)

    except Exception as e:
        print("Gagal membaca:", file)
        print("Error:", e)

# Gabungkan semua bagian
df = pd.concat(
    bagian_data,
    ignore_index=True,
    sort=False
)

# Rename kolom
NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}

df = df.rename(columns=NAMA_BARU)

# Pastikan wilayah berupa string
if "wilayah_jemput" in df.columns:
    df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)

print(f"baris: {len(df):,} | kolom: {df.shape[1]}")

# Ambil maksimal 300.000 data untuk proses praktikum
sample = (
    df.sample(
        n=min(300_000, len(df)),
        random_state=42
    )
    .reset_index(drop=True)
)

# Buat ID perjalanan
sample["id_perjalanan"] = sample.index + 1

# Waktu jemput dibuat string agar aman
if "waktu_jemput" in sample.columns:
    sample["waktu_jemput"] = sample["waktu_jemput"].astype(str)

print(f"sample: {len(sample):,} baris")
display(sample.head())

Jumlah file Parquet: 8
baris: 2,986,910 | kolom: 18
sample: 300,000 baris


,waktu_jemput,tpep_dropoff_datetime,passenger_count,jarak_mil,zona_jemput,zona_tujuan,payment_type,fare_amount,tip,total_bayar,durasi_menit,jam,metode_bayar,jam_mulai,suhu_c,hujan_mm,tanggal,wilayah_jemput,id_perjalanan
0,2023-01-16 10:40:39,2023-01-16 10:52:29,1.0,1.55,48,164,1,12.1,3.22,19.32,11.833333,10,Kartu kredit,2023-01-16 10:00:00,0.5,0.0,2023-01-16,Manhattan,1
1,2023-01-16 18:09:53,2023-01-16 18:16:40,2.0,0.57,164,170,1,7.2,2.24,13.44,6.783333,18,Kartu kredit,2023-01-16 18:00:00,3.2,0.0,2023-01-16,Manhattan,2
2,2023-01-18 15:29:58,2023-01-18 15:44:02,1.0,1.80,234,230,1,12.1,3.20,19.30,14.066667,15,Kartu kredit,2023-01-18 15:00:00,10.5,0.0,2023-01-18,Manhattan,3
3,2023-01-10 15:52:40,2023-01-10 16:14:36,1.0,3.04,48,114,1,21.2,5.04,30.24,21.933333,15,Kartu kredit,2023-01-10 15:00:00,6.3,0.0,2023-01-10,Manhattan,4
4,2023-01-24 14:11:55,2023-01-24 14:35:33,1.0,3.90,107,143,1,24.0,5.00,33.00,23.633333,14,Kartu kredit,2023-01-24 14:00:00,7.8,0.0,2023-01-24,Manhattan,5


# K-2 — Tabel biasa vs Key-value

**Dugaan:** tabel tanpa indeks paling lambat untuk pencarian ID; key-value dan tabel berindeks lebih cepat. SQL GROUP BY diperkirakan lebih cepat daripada membuka semua JSON.

In [9]:
KOLOM = ["id_perjalanan","waktu_jemput","zona_jemput","zona_tujuan",
          "wilayah_jemput","jarak_mil","total_bayar","metode_bayar"]
data = sample[KOLOM].copy()

if os.path.exists("uji.db"): os.remove("uji.db")
con = sqlite3.connect("uji.db")
ukur("relasional: muat tabel", lambda: data.to_sql("perjalanan", con, index=False))

def bangun_kv():
    return {f"perjalanan:{r['id_perjalanan']}": json.dumps(r)
            for r in data.to_dict("records")}
kv = ukur("key-value: bangun dict", bangun_kv)
print("jumlah key:", len(kv))

random.seed(42)
id_uji = random.sample(range(1, len(data)+1), 200)
def cari_sql():
    cur=con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?",(i,)).fetchone() for i in id_uji]
def cari_kv():
    return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji]

r1=ukur("cari 200 perjalanan - SQL tanpa indeks",cari_sql)
con.execute("CREATE INDEX idx_id ON perjalanan(id_perjalanan)")
r2=ukur("cari 200 perjalanan - SQL dengan indeks",cari_sql)
r3=ukur("cari 200 perjalanan - key-value",cari_kv)
print("hasil sama?", r1[0][0] == r2[0][0] == r3[0]["id_perjalanan"])

def hitung_sql():
    return con.execute("""SELECT wilayah_jemput, COUNT(*), ROUND(AVG(total_bayar),2)
                          FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC,1""").fetchall()
def hitung_kv():
    total,jumlah={},{}
    for teks in kv.values():
        d=json.loads(teks); w=d["wilayah_jemput"]
        total[w]=total.get(w,0)+d["total_bayar"]; jumlah[w]=jumlah.get(w,0)+1
    return sorted([(w,jumlah[w],round(total[w]/jumlah[w],2)) for w in jumlah],
                  key=lambda x:(-x[1],x[0]))
h1=ukur("hitung per wilayah - SQL GROUP BY",hitung_sql)
h2=ukur("hitung per wilayah - key-value (buka semua nilai)",hitung_kv)
print(h1[:3]); print("hasil sama?",h1==h2)

[relasional: muat tabel] 1.4385 s
[key-value: bangun dict] 4.5961 s
jumlah key: 300000
[cari 200 perjalanan - SQL tanpa indeks] 4.2438 s
[cari 200 perjalanan - SQL dengan indeks] 0.0041 s
[cari 200 perjalanan - key-value] 0.0012 s
hasil sama? True
[hitung per wilayah - SQL GROUP BY] 0.1975 s
[hitung per wilayah - key-value (buka semua nilai)] 1.2479 s
[('Manhattan', 267034, 22.7), ('Queens', 27288, 70.34), ('Unknown', 3787, 39.13)]
hasil sama? True


# K-3 — Document: bentuk data fleksibel

**Dugaan:** document lebih fleksibel ketika field opsional atau struktur berubah.

In [10]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen(r):
    dok={"id_perjalanan":int(r.id_perjalanan),
         "waktu_jemput":r.waktu_jemput,
         "jemput":{"zona":int(r.zona_jemput),"wilayah":r.wilayah_jemput},
         "tujuan":{"zona":int(r.zona_tujuan)},
         "biaya":{"jarak_mil":float(r.jarak_mil),"total":float(r.total_bayar),
                  "metode_bayar":r.metode_bayar}}
    if r.metode_bayar=="Kartu kredit":
        dok["biaya"]["tip"]=float(r.tip)
    return dok

db=TinyDB(storage=MemoryStorage)
tabel=db.table("perjalanan")
docs=[ke_dokumen(r) for r in sample.head(20_000).itertuples()]
tabel.insert_multiple(docs)
print("jumlah dokumen:",len(tabel))
print(json.dumps(tabel.all()[0],indent=2))

Q=Query()
print("jemput di Manhattan & total > 50 :",len(tabel.search((Q.jemput.wilayah=="Manhattan")&(Q.biaya.total>50))))
print("dokumen yang punya field tip :",len(tabel.search(Q.biaya.tip.exists())))
print("dokumen yang TIDAK punya tip :",len(tabel.search(~Q.biaya.tip.exists())))

tabel.insert({"id_perjalanan":999_999,"waktu_jemput":"2023-01-15 08:00:00",
              "jemput":{"zona":161,"wilayah":"Manhattan"},"tujuan":{"zona":236},
              "biaya":{"jarak_mil":1.2,"total":14.5,"metode_bayar":"Kartu kredit","tip":3.0},
              "cuaca":{"hujan":True,"suhu_c":-2.0}})
print("dokumen yang punya field cuaca :",len(tabel.search(Q.cuaca.exists())))
print("total dokumen sekarang :",len(tabel))

jumlah dokumen: 20000
{
  "id_perjalanan": 1,
  "waktu_jemput": "2023-01-16 10:40:39",
  "jemput": {
    "zona": 48,
    "wilayah": "Manhattan"
  },
  "tujuan": {
    "zona": 164
  },
  "biaya": {
    "jarak_mil": 1.55,
    "total": 19.32,
    "metode_bayar": "Kartu kredit",
    "tip": 3.22
  }
}
jemput di Manhattan & total > 50 : 693
dokumen yang punya field tip : 15906
dokumen yang TIDAK punya tip : 4094
dokumen yang punya field cuaca : 1
total dokumen sekarang : 20001


# K-4 — DuckDB + Parquet

**Dugaan:** DuckDB pada Parquet lebih cepat untuk agregasi karena column pruning dan pemrosesan analitik.

In [11]:
import duckdb
GLOB=f"{TRIPS_BERSIH}/*/*.parquet"
con_d=duckdb.connect()

con_d.sql(f"""CREATE VIEW perjalanan AS
SELECT tpep_pickup_datetime AS waktu_jemput,
       PULocationID AS zona_jemput,
       DOLocationID AS zona_tujuan,
       borough_naik AS wilayah_jemput,
       trip_distance AS jarak_mil,
       total_amount AS total_bayar,
       nama_pembayaran AS metode_bayar
FROM read_parquet('{GLOB}', hive_partitioning=true)""")

SQL="""SELECT wilayah_jemput,COUNT(*) AS jumlah,ROUND(AVG(total_bayar),2) AS rata_bayar
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY jumlah DESC,wilayah_jemput"""
hasil_d=ukur("DuckDB - hitung per wilayah (langsung dari Parquet)",lambda:con_d.sql(SQL).df())
print(hasil_d)

KOLOM_S=["wilayah_jemput","total_bayar","jarak_mil","zona_jemput","zona_tujuan"]
if os.path.exists("baris.db"): os.remove("baris.db")
con_s=sqlite3.connect("baris.db")
ukur("SQLite - muat seluruh baris ke tabel",lambda:df[KOLOM_S].to_sql("perjalanan",con_s,index=False))
hasil_s=ukur("SQLite - hitung per wilayah",lambda:con_s.execute(
    """SELECT wilayah_jemput,COUNT(*),ROUND(AVG(total_bayar),2)
       FROM perjalanan GROUP BY wilayah_jemput ORDER BY 2 DESC,1""").fetchall())
print("hasil sama?",[tuple(r) for r in hasil_d.itertuples(index=False)]==[tuple(r) for r in hasil_s])

SQL2="""SELECT COUNT(*),ROUND(AVG(total_bayar),2)
        FROM perjalanan WHERE wilayah_jemput='Staten Island'"""
print(ukur("DuckDB - hanya wilayah Staten Island",lambda:con_d.sql(SQL2).fetchall()))
ukuran_pq=sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(TRIPS_BERSIH) for f in fs)/1024**2
print(f"Parquet ({df.shape[1]} kolom): {ukuran_pq:.1f} MB | SQLite ({len(KOLOM_S)} kolom): {os.path.getsize('baris.db')/1024**2:.1f} MB")

[DuckDB - hitung per wilayah (langsung dari Parquet)] 0.2610 s
               wilayah_jemput   jumlah  rata_bayar
0                   Manhattan  2659609       22.72
1                      Queens   270315       70.46
2                     Unknown    37609       39.24
3                    Brooklyn    15724       32.72
4                       Bronx     3074       34.44
5  __HIVE_DEFAULT_PARTITION__      340       74.72
6               Staten Island      211       74.07
7                         EWR       28      106.41
[SQLite - muat seluruh baris ke tabel] 9.1017 s
[SQLite - hitung per wilayah] 2.1752 s
hasil sama? False
[DuckDB - hanya wilayah Staten Island] 0.0443 s
[(211, 74.07)]
Parquet (18 kolom): 66.5 MB | SQLite (5 kolom): 114.2 MB


# K-5 — Graph

**Dugaan:** graph lebih nyaman untuk pertanyaan keterhubungan multi-langkah karena traversal dapat memakai `cutoff`.

In [12]:
import networkx as nx
arus=ukur("hitung arus antar zona",lambda:con_d.sql(
    """SELECT zona_jemput AS asal,zona_tujuan AS tujuan,COUNT(*) AS jumlah
       FROM perjalanan GROUP BY 1,2""").df())

G=nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal),int(r.tujuan),jumlah=int(r.jumlah))
print("node (zona):",G.number_of_nodes(),"| edge (arus):",G.number_of_edges())

top_graf=sorted(G.edges(data=True),key=lambda e:(-e[2]["jumlah"],e[0],e[1]))[:5]
print("graph:",[(a,b,d["jumlah"]) for a,b,d in top_graf])
top_sql=arus.sort_values(["jumlah","asal","tujuan"],ascending=[False,True,True]).head(5)
print("SQL :",list(top_sql.itertuples(index=False,name=None)))
populer=sorted(G.in_degree(weight="jumlah"),key=lambda kv:(-kv[1],kv[0]))[:5]
print("zona tujuan terpopuler:",populer)

MIN=50
zona_awal=populer[0][0]
def dua_langkah_graph():
    hasil=set()
    for b,d1 in G[zona_awal].items():
        if d1["jumlah"]<MIN: continue
        for c,d2 in G[b].items():
            if d2["jumlah"]>=MIN and c!=zona_awal: hasil.add(c)
    return hasil
con_d.register("arus",arus)
def dua_langkah_sql():
    return set(con_d.sql(f"""SELECT DISTINCT a2.tujuan FROM arus a1
                              JOIN arus a2 ON a1.tujuan=a2.asal
                              WHERE a1.asal={zona_awal} AND a1.jumlah>={MIN}
                              AND a2.jumlah>={MIN} AND a2.tujuan<>{zona_awal}""").df()["tujuan"])
hg=ukur("2 langkah - graph (NetworkX)",dua_langkah_graph)
hs=ukur("2 langkah - SQL self-join (DuckDB)",dua_langkah_sql)
print("zona awal:",zona_awal,"| graph:",len(hg),"| SQL:",len(hs),"| sama?",hg==hs)

H=nx.DiGraph([(a,b) for a,b,d in G.edges(data=True) if d["jumlah"]>=MIN])
for k in (1,2,3,4):
    n=len(nx.single_source_shortest_path_length(H,zona_awal,cutoff=k))-1
    print(f"dalam <= {k} langkah: {n} zona")

[hitung arus antar zona] 1.7040 s
node (zona): 262 | edge (arus): 21583
graph: [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
SQL : [(237, 236, 22065), (236, 237, 18810), (236, 236, 14110), (237, 237, 13753), (264, 264, 13562)]
zona tujuan terpopuler: [(236, 144209), (237, 130369), (161, 113991), (230, 87706), (170, 87110)]
[2 langkah - graph (NetworkX)] 0.0083 s
[2 langkah - SQL self-join (DuckDB)] 0.0133 s
zona awal: 236 | graph: 214 | SQL: 214 | sama? True
dalam <= 1 langkah: 74 zona
dalam <= 2 langkah: 214 zona
dalam <= 3 langkah: 214 zona
dalam <= 4 langkah: 214 zona


# K-6 — Indeks spasial

**Dugaan:** STRtree lebih cepat daripada pemeriksaan semua titik, terutama untuk kotak kecil.

In [13]:
from shapely import STRtree, box, points
rng=np.random.default_rng(42)
N=200_000
bujur=rng.normal(-73.97,0.06,N).clip(-74.25,-73.70)
lintang=rng.normal(40.74,0.05,N).clip(40.50,40.92)
titik=points(bujur,lintang)
M=1000
cx=rng.uniform(-74.05,-73.85,M); cy=rng.uniform(40.65,40.85,M)
kotak=[box(x-0.005,y-0.005,x+0.005,y+0.005) for x,y in zip(cx,cy)]

def cara_loop(): return [sum(1 for p in titik if k.contains(p)) for k in kotak[:5]]
def cara_numpy():
    hasil=[]
    for x,y in zip(cx,cy):
        m=(bujur>=x-0.005)&(bujur<=x+0.005)&(lintang>=y-0.005)&(lintang<=y+0.005)
        hasil.append(int(m.sum()))
    return hasil

h_loop=ukur("cara 1 - loop Python, 5 kotak",cara_loop)
h_np=ukur("cara 2 - NumPy, 1000 kotak",cara_numpy)
pohon=ukur("bangun indeks STRtree",lambda:STRtree(titik))
h_idx=ukur("cara 3 - STRtree, 1000 kotak",lambda:[len(pohon.query(k,predicate="contains")) for k in kotak])
print("5 kotak pertama - loop:",h_loop,"| STRtree:",h_idx[:5])
print("hasil NumPy dan STRtree sama?",h_np==h_idx)
print("loop dan STRtree sama (5 pertama)?",h_loop==h_idx[:5])
print("total titik ditemukan:",sum(h_idx))

[cara 1 - loop Python, 5 kotak] 12.9054 s
[cara 2 - NumPy, 1000 kotak] 0.5625 s
[bangun indeks STRtree] 0.0744 s
[cara 3 - STRtree, 1000 kotak] 0.2007 s
5 kotak pertama - loop: [1012, 84, 228, 748, 509] | STRtree: [1012, 84, 228, 748, 509]
hasil NumPy dan STRtree sama? True
loop dan STRtree sama (5 pertama)? True
total titik ditemukan: 417545


# K-7 — Delta Lake: versi, time travel, schema enforcement

**Dugaan:** versi lama tetap dapat dibaca dan data dengan kolom asing ditolak.

In [15]:
from deltalake import write_deltalake, DeltaTable
import pandas as pd
import os
import shutil

PATH_DELTA = "/content/trips_delta"
PATH_DELTA_DRIVE = "/content/drive/MyDrive/BigData/Praktikum4/trips_delta"

# Hapus Delta table lama jika ada
if os.path.exists(PATH_DELTA):
    shutil.rmtree(PATH_DELTA)


data_delta = sample.copy()

data_awal = data_delta.iloc[:100_000].copy()

write_deltalake(
    PATH_DELTA,
    data_awal,
    mode="overwrite"
)

dt = DeltaTable(PATH_DELTA)

print("Versi awal :", dt.version())
print("Jumlah baris:", len(dt.to_pandas()))


data_tambahan = data_delta.iloc[100_000:150_000].copy()

write_deltalake(
    PATH_DELTA,
    data_tambahan,
    mode="append"
)

dt = DeltaTable(PATH_DELTA)

print("\nSetelah append:")
print("Versi :", dt.version())
print("Jumlah baris:", len(dt.to_pandas()))


data_sekarang = dt.to_pandas()

kondisi_update = (
    (data_sekarang["wilayah_jemput"].astype(str) == "Bronx") &
    (data_sekarang["metode_bayar"].astype(str) == "Tunai")
)

jumlah_update = kondisi_update.sum()

data_sekarang.loc[
    kondisi_update,
    "total_bayar"
] = (
    pd.to_numeric(
        data_sekarang.loc[kondisi_update, "total_bayar"],
        errors="coerce"
    ) + 1.0
)

# Tulis kembali hasil update sebagai versi baru
write_deltalake(
    PATH_DELTA,
    data_sekarang,
    mode="overwrite"
)

dt = DeltaTable(PATH_DELTA)

print("\nSetelah update:")
print("Versi :", dt.version())
print("Baris yang di-update:", jumlah_update)


try:
    v0 = DeltaTable(
        PATH_DELTA,
        version=0
    ).to_pandas()

    print("\nVersion 0:")
    print("Jumlah baris:", len(v0))

except Exception as e:
    print("Version 0 tidak tersedia:", e)


try:
    v1 = DeltaTable(
        PATH_DELTA,
        version=1
    ).to_pandas()

    print("\nVersion 1:")
    print("Jumlah baris:", len(v1))

except Exception as e:
    print("Version 1 tidak tersedia:", e)


terbaru = DeltaTable(PATH_DELTA).to_pandas()

print("\nData terbaru:")
print("Jumlah baris:", len(terbaru))

if "total_bayar" in terbaru.columns:
    print(
        "Total bayar:",
        round(
            pd.to_numeric(
                terbaru["total_bayar"],
                errors="coerce"
            ).sum(),
            2
        )
    )


print("\n==============================")
print("DELTA HISTORY")
print("==============================")

dt = DeltaTable(PATH_DELTA)

for h in dt.history():
    print(
        "version:",
        h.get("version"),
        "| operation:",
        h.get("operation")
    )



data_schema_salah = data_delta.iloc[:10].copy()

data_schema_salah["kolom_asing"] = 1

try:
    write_deltalake(
        PATH_DELTA,
        data_schema_salah,
        mode="append"
    )

    print(
        "\nAppend dengan kolom asing berhasil."
    )

except Exception as e:
    print(
        "\nAppend kolom asing DITOLAK "
        "(schema enforcement):"
    )
    print(type(e).__name__, ":", e)


if os.path.exists(PATH_DELTA_DRIVE):
    shutil.rmtree(PATH_DELTA_DRIVE)

shutil.copytree(
    PATH_DELTA,
    PATH_DELTA_DRIVE
)

print("\nSalinan Delta Lake disimpan di:")
print(PATH_DELTA_DRIVE)

print("\nK-7 selesai.")

Versi awal : 0
Jumlah baris: 100000

Setelah append:
Versi : 1
Jumlah baris: 150000

Setelah update:
Versi : 2
Baris yang di-update: 20

Version 0:
Jumlah baris: 100000

Version 1:
Jumlah baris: 150000

Data terbaru:
Jumlah baris: 150000
Total bayar: 4100192.91

DELTA HISTORY
version: 2 | operation: WRITE
version: 1 | operation: WRITE
version: 0 | operation: WRITE

Append kolom asing DITOLAK (schema enforcement):
SchemaMismatchError : Cannot cast schema, number of fields does not match: 20 vs 19

Salinan Delta Lake disimpan di:
/content/drive/MyDrive/BigData/Praktikum4/trips_delta

K-7 selesai.


# K-8 — Small File Problem dan Compaction

**Dugaan:** 20 append kecil menaikkan file aktif; compaction menurunkannya tanpa mengubah jumlah baris.

In [16]:
dt=DeltaTable(PATH_DELTA)
print("file data aktif sebelum:",len(dt.file_uris()),"| versi:",dt.version())

for i in range(20):
    potongan=data_delta.iloc[150_000+i*500:150_000+(i+1)*500]
    write_deltalake(PATH_DELTA,potongan,mode="append")

dt=DeltaTable(PATH_DELTA)
print("file data aktif setelah 20 append kecil:",len(dt.file_uris()),"| versi:",dt.version(),"| baris:",len(dt.to_pandas()))
t0=time.perf_counter(); _=DeltaTable(PATH_DELTA).to_pandas(); t_sebelum=time.perf_counter()-t0

hasil=dt.optimize.compact()
print("hasil compact:",{k:hasil[k] for k in ("numFilesAdded","numFilesRemoved")})

dt=DeltaTable(PATH_DELTA)
t0=time.perf_counter(); _=DeltaTable(PATH_DELTA).to_pandas(); t_sesudah=time.perf_counter()-t0
print("file data aktif setelah compact:",len(dt.file_uris()),"| versi:",dt.version(),"| baris:",len(dt.to_pandas()))
print(f"waktu baca sebelum compact: {t_sebelum:.4f} s | sesudah: {t_sesudah:.4f} s")
print("file parquet di disk:",len(glob.glob(f"{PATH_DELTA}/*.parquet")))
kandidat=dt.vacuum(retention_hours=0,enforce_retention_duration=False,dry_run=True)
print("file yang akan dihapus VACUUM (dry run):",len(kandidat))

file data aktif sebelum: 1 | versi: 2
file data aktif setelah 20 append kecil: 21 | versi: 22 | baris: 160000
hasil compact: {'numFilesAdded': 1, 'numFilesRemoved': 21}
file data aktif setelah compact: 1 | versi: 23 | baris: 160000
waktu baca sebelum compact: 0.1981 s | sesudah: 0.2342 s
file parquet di disk: 24
file yang akan dihapus VACUUM (dry run): 23


# K-9 — Merangkum dan Memilih

In [17]:
rekap=pd.DataFrame(catatan)
rekap.to_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv",index=False)
print(rekap.to_string(index=False))

                                            langkah   detik
                             relasional: muat tabel  1.4385
                             key-value: bangun dict  4.5961
             cari 200 perjalanan - SQL tanpa indeks  4.2438
            cari 200 perjalanan - SQL dengan indeks  0.0041
                    cari 200 perjalanan - key-value  0.0012
                  hitung per wilayah - SQL GROUP BY  0.1975
  hitung per wilayah - key-value (buka semua nilai)  1.2479
DuckDB - hitung per wilayah (langsung dari Parquet)  0.2610
               SQLite - muat seluruh baris ke tabel  9.1017
                        SQLite - hitung per wilayah  2.1752
               DuckDB - hanya wilayah Staten Island  0.0443
                             hitung arus antar zona  1.7040
                       2 langkah - graph (NetworkX)  0.0083
                 2 langkah - SQL self-join (DuckDB)  0.0133
                      cara 1 - loop Python, 5 kotak 12.9054
                         cara 2 - NumPy,

# O. ANALISIS HASIL

### 1. Pencarian satu data

Pada pencarian 200 perjalanan, SQL tanpa indeks membutuhkan waktu 4,2438 detik, sedangkan SQL dengan indeks hanya 0,0041 detik. Dengan demikian, tabel tanpa indeks sekitar 1.035,07 kali lebih lambat dibandingkan tabel berindeks.

Sementara itu, SQL dengan indeks membutuhkan 0,0041 detik dan key-value membutuhkan 0,0012 detik. Artinya, tabel berindeks sekitar 3,42 kali lebih lambat dibandingkan key-value.

Perbedaan yang paling besar terlihat pada perbandingan tabel tanpa indeks dengan tabel berindeks karena pencarian tanpa indeks harus memeriksa data satu per satu. Penggunaan indeks membuat pencarian berdasarkan ID menjadi jauh lebih cepat.

### 2. Harga kesederhanaan

Untuk menghitung rata-rata total pembayaran per wilayah, SQL GROUP BY membutuhkan waktu 0,1975 detik, sedangkan key-value membutuhkan 1,2479 detik. Key-value sekitar 6,32 kali lebih lambat dibandingkan SQL GROUP BY.

Hal ini terjadi karena pada key-value setiap nilai harus dibuka dan diperiksa satu per satu sebelum dilakukan perhitungan. Key-value sederhana dan sangat cepat untuk pencarian berdasarkan key, tetapi kurang efisien untuk agregasi yang membutuhkan pemeriksaan banyak data.

### 3. Data fleksibel

Pada model document, terdapat 15.906 dokumen yang memiliki field tip dan 4.094 dokumen yang tidak memiliki field tip. Selain itu, satu dokumen berhasil ditambahkan dengan field baru berupa cuaca tanpa harus mengubah struktur dokumen lainnya.

Keuntungannya adalah model document lebih fleksibel ketika sebagian data memiliki field tambahan atau struktur yang berubah. Risikonya adalah validasi struktur data menjadi lebih bergantung pada aplikasi sehingga kemungkinan ketidakkonsistenan data menjadi lebih besar.

### 4. Penyimpanan per kolom

DuckDB membutuhkan waktu 0,2610 detik untuk menghitung data per wilayah, sedangkan SQLite membutuhkan waktu 2,1752 detik untuk perhitungan yang sama. Berdasarkan waktu tersebut, DuckDB sekitar 8,33 kali lebih cepat daripada SQLite.

Keunggulan tersebut dapat dijelaskan oleh column pruning, yaitu DuckDB dapat membaca hanya kolom yang dibutuhkan oleh query. Selain itu, DuckDB melakukan pemrosesan data secara efisien dan dapat memproses banyak baris secara sekaligus.

Pada filter wilayah Staten Island, DuckDB hanya membutuhkan 0,0443 detik. Namun, pada percobaan ini hasil DuckDB dan SQLite tercatat `False`, sehingga hasil agregasi kedua engine tidak sepenuhnya sama. Karena itu, perbandingan waktu tetap dapat diamati, tetapi hasil agregasinya perlu diperiksa kembali sebelum digunakan sebagai pembuktian kesamaan hasil.

Ukuran penyimpanan Parquet adalah 66,5 MB, sedangkan SQLite yang digunakan berukuran 114,2 MB.

### 5. Graph

Pada pencarian dua langkah, graph membutuhkan waktu 0,0083 detik, sedangkan SQL self-join membutuhkan waktu 0,0133 detik. Graph sekitar 1,60 kali lebih cepat pada percobaan ini.

Hasil graph dan SQL sama-sama menghasilkan 214 zona, sehingga hasil keterhubungannya konsisten. Selain kecepatan, graph tetap layak digunakan karena pertanyaan keterhubungan dengan jumlah langkah yang berbeda lebih alami dilakukan dengan penelusuran graph. Hasil percobaan menunjukkan 74 zona dapat dicapai dalam paling banyak satu langkah, 214 zona dalam paling banyak dua langkah, dan tetap 214 zona pada tiga maupun empat langkah.

### 6. Indeks spasial

Pada percobaan utama, NumPy membutuhkan 0,5625 detik untuk memeriksa 1.000 kotak, sedangkan STRtree membutuhkan 0,2007 detik. Dengan demikian, STRtree sekitar 2,80 kali lebih cepat daripada NumPy.

Pada latihan dengan ukuran kotak yang berbeda, untuk sisi 0,02 derajat NumPy membutuhkan 0,541454 detik dan STRtree 0,774006 detik, sehingga pada ukuran kotak tersebut STRtree justru lebih lambat. Untuk sisi 0,001 derajat, NumPy membutuhkan 0,593303 detik sedangkan STRtree hanya 0,018462 detik. Artinya, pada kotak yang lebih kecil STRtree sekitar 32,14 kali lebih cepat daripada NumPy.

Hal tersebut menunjukkan bahwa keuntungan indeks spasial bergantung pada bentuk dan ukuran wilayah pencarian. Kotak yang lebih kecil memungkinkan indeks mengurangi lebih banyak kandidat titik yang perlu diperiksa.

# LATIHAN PRAKTIKUM

# Q1 — 2.000 ID acak
Hitung rata-rata waktu per pencarian dalam mikrodetik.

In [18]:
random.seed(42)
id_uji_2000=random.sample(range(1,len(data)+1),2000)
def cari_sql_2000():
    cur=con.cursor()
    return [cur.execute("SELECT * FROM perjalanan WHERE id_perjalanan = ?",(i,)).fetchone() for i in id_uji_2000]
def cari_kv_2000(): return [json.loads(kv[f"perjalanan:{i}"]) for i in id_uji_2000]
t0=time.perf_counter(); _=cari_sql_2000(); w_sql=time.perf_counter()-t0
t0=time.perf_counter(); _=cari_kv_2000(); w_kv=time.perf_counter()-t0
print(f"SQL berindeks: {w_sql/2000*1_000_000:.3f} mikrodetik/pencarian")
print(f"Key-value: {w_kv/2000*1_000_000:.3f} mikrodetik/pencarian")

SQL berindeks: 22.732 mikrodetik/pencarian
Key-value: 16.810 mikrodetik/pencarian


# Q2 — Document vs SQL
Hitung dokumen dengan zona tujuan tertentu dan metode bayar Tunai, lalu bandingkan dengan SQL.

In [19]:
zona_target=int(data.iloc[0]["zona_tujuan"])
jumlah_doc=len(tabel.search((Q.tujuan.zona==zona_target)&(Q.biaya.metode_bayar=="Tunai")))
jumlah_sql=con.execute("SELECT COUNT(*) FROM perjalanan WHERE zona_tujuan=? AND metode_bayar='Tunai'",(zona_target,)).fetchone()[0]
print("zona target:",zona_target)
print("document:",jumlah_doc,"| SQL:",jumlah_sql,"| sama?",jumlah_doc==jumlah_sql)

zona target: 164
document: 93 | SQL: 1086 | sama? False


# Q3 — SUM satu kolom vs beberapa kolom

In [20]:
ukur("DuckDB - SUM satu kolom",lambda:con_d.sql("SELECT SUM(total_bayar) FROM perjalanan").fetchall())
ukur("DuckDB - SUM beberapa kolom",lambda:con_d.sql(
    """SELECT SUM(total_bayar),SUM(jarak_mil),SUM(zona_jemput),SUM(zona_tujuan) FROM perjalanan"""
).fetchall())

[DuckDB - SUM satu kolom] 0.4821 s
[DuckDB - SUM beberapa kolom] 0.8149 s


[(81621442.45996648, 10231889.610000132, 497452755, 491330400)]

# Q4 — Lima zona dengan out-degree terbesar

In [21]:
out_graph=sorted([(n,G.out_degree(n)) for n in G.nodes()],key=lambda x:(-x[1],x[0]))[:5]
out_sql=con_d.sql("""SELECT zona_jemput AS zona,COUNT(DISTINCT zona_tujuan) AS jumlah_tujuan
                     FROM perjalanan GROUP BY zona_jemput
                     ORDER BY jumlah_tujuan DESC,zona LIMIT 5""").df()
print("Graph:",out_graph)
print("SQL:")
print(out_sql)

Graph: [(132, 260), (138, 254), (264, 241), (230, 234), (140, 233)]
SQL:
   zona  jumlah_tujuan
0   132            260
1   138            254
2   264            241
3   230            234
4   140            233


# Q5 — Ukuran kotak 0,02 dan 0,001 derajat

In [22]:
def ukur_spasial(sisi):
    kotak_uji=[box(x-sisi/2,y-sisi/2,x+sisi/2,y+sisi/2) for x,y in zip(cx,cy)]
    t0=time.perf_counter()
    hasil_np=[]
    for x,y in zip(cx,cy):
        m=(bujur>=x-sisi/2)&(bujur<=x+sisi/2)&(lintang>=y-sisi/2)&(lintang<=y+sisi/2)
        hasil_np.append(int(m.sum()))
    w_np=time.perf_counter()-t0
    t0=time.perf_counter()
    hasil_idx=[len(pohon.query(k,predicate="contains")) for k in kotak_uji]
    w_idx=time.perf_counter()-t0
    return {"sisi":sisi,"numpy":w_np,"strtree":w_idx,"rasio_numpy_strtree":w_np/w_idx,"sama":hasil_np==hasil_idx}
print(pd.DataFrame([ukur_spasial(0.02),ukur_spasial(0.001)]).to_string(index=False))

 sisi    numpy  strtree  rasio_numpy_strtree  sama
0.020 0.541454 0.774006             0.699548  True
0.001 0.593303 0.018462            32.136833  True


# Q6 — Time Travel vs Restore

In [23]:
dt=DeltaTable(PATH_DELTA)
print("versi sebelum restore:",dt.version(),"| baris:",len(dt.to_pandas()))
dt.restore(1)
dt=DeltaTable(PATH_DELTA)
print("versi setelah restore:",dt.version(),"| baris:",len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 23 | baris: 160000
versi setelah restore: 24 | baris: 150000
['RESTORE', 'OPTIMIZE', 'WRITE']


# R — Tugas Praktikum

# R1. Rekap Benchmark

Berdasarkan hasil pengukuran, benchmark menunjukkan bahwa setiap model penyimpanan memiliki keunggulan pada jenis pertanyaan tertentu. Pencarian dengan key-value dan tabel berindeks jauh lebih cepat dibandingkan tabel tanpa indeks, sedangkan DuckDB lebih sesuai untuk perhitungan agregasi. Graph digunakan untuk pertanyaan keterhubungan dan STRtree digunakan untuk pencarian spasial.

In [24]:
# R1 — Rekap benchmark + interpretasi

rekap_r1 = pd.DataFrame(catatan)

def interpretasi(label, waktu):
    label = str(label).lower()

    if "tanpa indeks" in label:
        return "Pencarian tanpa indeks membutuhkan pemeriksaan data lebih banyak sehingga waktunya paling lambat."

    elif "dengan indeks" in label:
        return "Indeks mempercepat pencarian berdasarkan ID karena lokasi data dapat ditemukan lebih langsung."

    elif "key-value" in label and "bangun" in label:
        return "Pembuatan struktur key-value membutuhkan waktu karena seluruh data harus dimasukkan ke dalam dictionary."

    elif "key-value" in label and "hitung" in label:
        return "Key-value lebih lambat untuk agregasi karena seluruh nilai perlu dibuka dan diperiksa."

    elif "key-value" in label and "cari" in label:
        return "Key-value sangat cepat untuk pencarian berdasarkan key."

    elif "duckdb" in label:
        return "DuckDB efisien untuk analisis data karena dapat membaca dan memproses kolom yang diperlukan."

    elif "sqlite" in label and "muat" in label:
        return "Pemuatan seluruh data ke SQLite membutuhkan waktu cukup besar."

    elif "sqlite" in label and "hitung" in label:
        return "SQLite dapat melakukan agregasi, tetapi pada percobaan ini lebih lambat dibandingkan DuckDB."

    elif "graph" in label:
        return "Graph efektif digunakan untuk mencari keterhubungan antarzona."

    elif "self-join" in label:
        return "SQL self-join dapat menjawab keterhubungan, tetapi membutuhkan operasi JOIN antar tabel arus."

    elif "loop python" in label:
        return "Pemeriksaan titik satu per satu membutuhkan waktu paling besar."

    elif "numpy" in label:
        return "NumPy mempercepat pemeriksaan spasial dibandingkan loop Python dengan operasi array."

    elif "strtree" in label:
        return "STRtree dapat mempercepat pencarian spasial dengan menggunakan indeks lokasi."

    else:
        return "Waktu menunjukkan biaya proses penyimpanan atau pengolahan data."

rekap_r1["interpretasi"] = [
    interpretasi(row["langkah"], row["detik"])
    for _, row in rekap_r1.iterrows()
]

display(rekap_r1)

# Simpan kembali
rekap_r1.to_csv(
    f"{DIR_SIMPAN}/benchmark_nosql_dengan_interpretasi.csv",
    index=False
)

print("File tersimpan.")

,langkah,detik,interpretasi
0,relasional: muat tabel,1.4385,Waktu menunjukkan biaya proses penyimpanan ata...
1,key-value: bangun dict,4.5961,Pembuatan struktur key-value membutuhkan waktu...
2,cari 200 perjalanan - SQL tanpa indeks,4.2438,Pencarian tanpa indeks membutuhkan pemeriksaan...
3,cari 200 perjalanan - SQL dengan indeks,0.0041,Indeks mempercepat pencarian berdasarkan ID ka...
4,cari 200 perjalanan - key-value,0.0012,Key-value sangat cepat untuk pencarian berdasa...
5,hitung per wilayah - SQL GROUP BY,0.1975,Waktu menunjukkan biaya proses penyimpanan ata...
6,hitung per wilayah - key-value (buka semua nilai),1.2479,Key-value lebih lambat untuk agregasi karena s...
7,DuckDB - hitung per wilayah (langsung dari Par...,0.2610,DuckDB efisien untuk analisis data karena dapa...
8,SQLite - muat seluruh baris ke tabel,9.1017,Pemuatan seluruh data ke SQLite membutuhkan wa...
9,SQLite - hitung per wilayah,2.1752,"SQLite dapat melakukan agregasi, tetapi pada p..."


File tersimpan.


# R2. Data Lain, Cara Lain

Data yang dipilih adalah **metode pembayaran**. Data ini memiliki beberapa kategori seperti Kartu kredit, Tunai, Gratis, dan Sengketa.

### 1. Model Tabel

Data metode pembayaran disimpan dalam bentuk tabel dengan kolom seperti `id_perjalanan` dan `metode_bayar`.

**Pertanyaan yang mudah dijawab:**
Berapa jumlah perjalanan untuk setiap metode pembayaran?

**Pertanyaan yang canggung dijawab:**
Dari satu metode pembayaran tertentu, zona mana saja yang saling terhubung dalam tiga langkah perjalanan?

### 2. Model Document

Data perjalanan disimpan sebagai dokumen dengan informasi pembayaran berada di dalam objek `biaya`, misalnya `metode_bayar`, `total`, dan `tip`.

**Pertanyaan yang mudah dijawab:**
Tampilkan semua perjalanan yang menggunakan metode pembayaran tertentu dan memiliki total pembayaran di atas nilai tertentu.

**Pertanyaan yang canggung dijawab:**
Hitung hubungan perjalanan antarzona berdasarkan metode pembayaran untuk beberapa tingkat keterhubungan.

### 3. Model Key-Value

Setiap perjalanan disimpan menggunakan `id_perjalanan` sebagai key dan data perjalanan sebagai value.

**Pertanyaan yang mudah dijawab:**
Tampilkan detail pembayaran berdasarkan ID perjalanan tertentu.

**Pertanyaan yang canggung dijawab:**
Hitung rata-rata total pembayaran untuk setiap metode pembayaran dengan memeriksa seluruh data.

In [25]:
# R2 — Contoh pemodelan data metode pembayaran

r2 = sample[
    ["id_perjalanan", "metode_bayar", "total_bayar", "zona_jemput", "zona_tujuan"]
].copy()

print("Jumlah data:", len(r2))

print("\n--- MODEL TABEL ---")
display(
    r2.groupby("metode_bayar")
      .agg(
          jumlah_perjalanan=("id_perjalanan", "count"),
          rata_rata_bayar=("total_bayar", "mean")
      )
      .reset_index()
)

print("\n--- MODEL DOCUMENT ---")

dokumen_r2 = []

for _, row in r2.head(5).iterrows():
    dokumen_r2.append({
        "id_perjalanan": int(row["id_perjalanan"]),
        "perjalanan": {
            "zona_jemput": int(row["zona_jemput"]),
            "zona_tujuan": int(row["zona_tujuan"])
        },
        "pembayaran": {
            "metode": row["metode_bayar"],
            "total": float(row["total_bayar"])
        }
    })

display(dokumen_r2)

print("\n--- MODEL KEY-VALUE ---")

kv_r2 = {
    f"perjalanan:{int(row.id_perjalanan)}": {
        "metode_bayar": row.metode_bayar,
        "total_bayar": float(row.total_bayar)
    }
    for row in r2.head(10).itertuples()
}

for k, v in list(kv_r2.items())[:3]:
    print(k, "=>", v)

Jumlah data: 300000

--- MODEL TABEL ---


,metode_bayar,jumlah_perjalanan,rata_rata_bayar
0,Gratis,943,22.913075
1,Kartu kredit,239237,28.118488
2,Sengketa,1545,24.609120
3,Tunai,51898,23.476644



--- MODEL DOCUMENT ---


[{'id_perjalanan': 1,
  'perjalanan': {'zona_jemput': 48, 'zona_tujuan': 164},
  'pembayaran': {'metode': 'Kartu kredit', 'total': 19.32}},
 {'id_perjalanan': 2,
  'perjalanan': {'zona_jemput': 164, 'zona_tujuan': 170},
  'pembayaran': {'metode': 'Kartu kredit', 'total': 13.44}},
 {'id_perjalanan': 3,
  'perjalanan': {'zona_jemput': 234, 'zona_tujuan': 230},
  'pembayaran': {'metode': 'Kartu kredit', 'total': 19.3}},
 {'id_perjalanan': 4,
  'perjalanan': {'zona_jemput': 48, 'zona_tujuan': 114},
  'pembayaran': {'metode': 'Kartu kredit', 'total': 30.24}},
 {'id_perjalanan': 5,
  'perjalanan': {'zona_jemput': 107, 'zona_tujuan': 143},
  'pembayaran': {'metode': 'Kartu kredit', 'total': 33.0}}]


--- MODEL KEY-VALUE ---
perjalanan:1 => {'metode_bayar': 'Kartu kredit', 'total_bayar': 19.32}
perjalanan:2 => {'metode_bayar': 'Kartu kredit', 'total_bayar': 13.44}
perjalanan:3 => {'metode_bayar': 'Kartu kredit', 'total_bayar': 19.3}


# R3 — Contoh simulasi Delta Lake dua hari

In [26]:
# ============================================================
# R3 — DELTA LAKE DUA HARI
# ============================================================

from deltalake import write_deltalake, DeltaTable
import pandas as pd
import os
import shutil
import json

# Gunakan folder khusus R3
PATH_R3 = "/content/trips_delta_R3"
PATH_R3_DRIVE = f"{DIR_SIMPAN}/trips_delta_R3"

# Bersihkan percobaan lama
if os.path.exists(PATH_R3):
    shutil.rmtree(PATH_R3)

if os.path.exists(PATH_R3_DRIVE):
    shutil.rmtree(PATH_R3_DRIVE)

# ------------------------------------------------------------
# HARI 1
# ------------------------------------------------------------

data_r3 = sample.copy()

hari1 = data_r3.iloc[:100_000].copy()

write_deltalake(
    PATH_R3,
    hari1,
    mode="overwrite"
)

dt = DeltaTable(PATH_R3)

print("========== HARI 1 ==========")
print("Versi:", dt.version())
print("Jumlah baris:", len(dt.to_pandas()))


# ------------------------------------------------------------
# HARI 2 — tambah data
# ------------------------------------------------------------

hari2_tambahan = data_r3.iloc[100_000:150_000].copy()

write_deltalake(
    PATH_R3,
    hari2_tambahan,
    mode="append"
)

dt = DeltaTable(PATH_R3)

print("\n========== HARI 2 - APPEND ==========")
print("Versi:", dt.version())
print("Jumlah baris:", len(dt.to_pandas()))


# ------------------------------------------------------------
# HARI 2 — koreksi beberapa baris
# ------------------------------------------------------------

data_sekarang = dt.to_pandas()

# Pilih 10 baris pertama untuk dikoreksi
jumlah_koreksi = 10

indeks_koreksi = data_sekarang.index[:jumlah_koreksi]

nilai_sebelum = data_sekarang.loc[
    indeks_koreksi,
    "total_bayar"
].copy()

data_sekarang.loc[
    indeks_koreksi,
    "total_bayar"
] = (
    pd.to_numeric(
        data_sekarang.loc[indeks_koreksi, "total_bayar"],
        errors="coerce"
    ) + 1.0
)

write_deltalake(
    PATH_R3,
    data_sekarang,
    mode="overwrite"
)

dt = DeltaTable(PATH_R3)

print("\n========== HARI 2 - KOREKSI ==========")
print("Versi:", dt.version())
print("Jumlah baris:", len(dt.to_pandas()))
print("Jumlah baris dikoreksi:", jumlah_koreksi)


# ------------------------------------------------------------
# BUKTI TIME TRAVEL
# ------------------------------------------------------------

akhir_hari1 = DeltaTable(
    PATH_R3,
    version=0
).to_pandas()

kondisi_akhir = DeltaTable(PATH_R3).to_pandas()

print("\n========== TIME TRAVEL ==========")
print("Jumlah baris akhir Hari 1:", len(akhir_hari1))
print("Jumlah baris kondisi akhir:", len(kondisi_akhir))

print(
    "\nTime travel berhasil:",
    len(akhir_hari1) == 100_000
)


# ------------------------------------------------------------
# Tampilkan beberapa data versi Hari 1
# ------------------------------------------------------------

print("\nContoh data akhir Hari 1:")
display(akhir_hari1.head())


# ------------------------------------------------------------
# HISTORY
# ------------------------------------------------------------

print("\n========== DELTA HISTORY ==========")

dt = DeltaTable(PATH_R3)

for h in dt.history():
    print(
        "version:",
        h.get("version"),
        "| operation:",
        h.get("operation")
    )


# ------------------------------------------------------------
# ISI _delta_log
# ------------------------------------------------------------

DELTA_LOG = os.path.join(
    PATH_R3,
    "_delta_log"
)

file_log = sorted(
    os.listdir(DELTA_LOG)
)

print("\n========== ISI _delta_log ==========")
print("Jumlah file log:", len(file_log))

for nama in file_log:
    print(nama)


# Tampilkan isi JSON log
print("\n========== CONTOH ISI LOG ==========")

for nama in file_log:
    if nama.endswith(".json"):
        print("\nFile:", nama)

        with open(
            os.path.join(DELTA_LOG, nama),
            "r"
        ) as f:

            for i, baris in enumerate(f):
                print(baris.strip())

                if i >= 4:
                    break

        break


# ------------------------------------------------------------
# ZIP _delta_log + data Delta
# ------------------------------------------------------------

shutil.copytree(
    PATH_R3,
    PATH_R3_DRIVE
)

print("\nFolder R3 disalin ke:")
print(PATH_R3_DRIVE)

print("\nR3 selesai.")

========== HARI 1 ==========
Versi: 0
Jumlah baris: 100000

========== HARI 2 - APPEND ==========
Versi: 1
Jumlah baris: 150000

========== HARI 2 - KOREKSI ==========
Versi: 2
Jumlah baris: 150000
Jumlah baris dikoreksi: 10

========== TIME TRAVEL ==========
Jumlah baris akhir Hari 1: 100000
Jumlah baris kondisi akhir: 150000

Time travel berhasil: True

Contoh data akhir Hari 1:


,waktu_jemput,tpep_dropoff_datetime,passenger_count,jarak_mil,zona_jemput,zona_tujuan,payment_type,fare_amount,tip,total_bayar,durasi_menit,jam,metode_bayar,jam_mulai,suhu_c,hujan_mm,tanggal,wilayah_jemput,id_perjalanan
0,2023-01-16 10:40:39,2023-01-16 10:52:29,1.0,1.55,48,164,1,12.1,3.22,19.32,11.833333,10,Kartu kredit,2023-01-16 10:00:00,0.5,0.0,2023-01-16,Manhattan,1
1,2023-01-16 18:09:53,2023-01-16 18:16:40,2.0,0.57,164,170,1,7.2,2.24,13.44,6.783333,18,Kartu kredit,2023-01-16 18:00:00,3.2,0.0,2023-01-16,Manhattan,2
2,2023-01-18 15:29:58,2023-01-18 15:44:02,1.0,1.80,234,230,1,12.1,3.20,19.30,14.066667,15,Kartu kredit,2023-01-18 15:00:00,10.5,0.0,2023-01-18,Manhattan,3
3,2023-01-10 15:52:40,2023-01-10 16:14:36,1.0,3.04,48,114,1,21.2,5.04,30.24,21.933333,15,Kartu kredit,2023-01-10 15:00:00,6.3,0.0,2023-01-10,Manhattan,4
4,2023-01-24 14:11:55,2023-01-24 14:35:33,1.0,3.90,107,143,1,24.0,5.00,33.00,23.633333,14,Kartu kredit,2023-01-24 14:00:00,7.8,0.0,2023-01-24,Manhattan,5



========== DELTA HISTORY ==========
version: 2 | operation: WRITE
version: 1 | operation: WRITE
version: 0 | operation: WRITE

========== ISI _delta_log ==========
Jumlah file log: 3
00000000000000000000.json
00000000000000000001.json
00000000000000000002.json

========== CONTOH ISI LOG ==========

File: 00000000000000000000.json
{"commitInfo":{"timestamp":1791201245075,"operation":"WRITE","operationParameters":{"mode":"Overwrite"},"engineInfo":"delta-rs:py-1.6.6","operationMetrics":{"num_added_files":1,"num_removed_files":0,"num_partitions":0,"num_added_rows":100000,"execution_time_ms":125,"num_retries":0},"clientVersion":"delta-rs.py-1.6.6"}}
{"protocol":{"minReaderVersion":3,"minWriterVersion":7,"readerFeatures":["timestampNtz"],"writerFeatures":["timestampNtz"]}}
{"metaData":{"id":"3f80b4bd-933d-4f65-bf49-5ccbed0b87a1","name":null,"description":null,"format":{"provider":"parquet","options":{}},"schemaString":"{\"type\":\"struct\",\"fields\":[{\"name\":\"waktu_jemput\",\"type\":\"s